In [7]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
from s0_fun_base import XY_flow_compu
import gpflow
from s0_class_GP_IPM import Perted_IPM
from tensorflow_probability import distributions as tfd
f64 = gpflow.utilities.to_default_float
from sklearn.metrics import roc_curve, auc

In [ ]:
convert_to_constrained_values = 'ON'
truedata_style='glm'
grw_setting='sep'
target='flow'

if convert_to_constrained_values == 'ON':
    # converting unconstrained values to constrained space & calculating the likelihood values.
    true_population = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/"+truedata_style+"_mle_true_population.pkl", mode="rb"))
    df_flow = XY_flow_compu(true_population)  

    m_flow_new = gpflow.models.GPMC(data=(df_flow[0], df_flow[1]), 
                                    kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Poisson())

    # Secondly, we add priors to the hyperparameters.
    m_flow_new.kernel.lengthscales.prior = tfd.HalfNormal(scale=f64(100.))
    m_flow_new.kernel.variance.prior = tfd.HalfNormal(scale=f64(100.))

    # We now sample from the posterior using HMC.
    hmc_helper = gpflow.optimizers.SamplingHelper(
        m_flow_new.log_posterior_density, m_flow_new.trainable_parameters
    )

    samples = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/samples_"+target+".pkl", mode="rb")) 
    constrained_samples = hmc_helper.convert_to_constrained_values(samples)
    pickle.dump(constrained_samples, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/constrained_samples_"+target+".pkl", mode="wb"))

    nll = np.array([])
    nlp = np.array([])

    for i in range(5000):
        for var, var_samples in zip(hmc_helper.current_state, samples):
            var.assign(var_samples[i])
        
        nll = np.append(nll, np.array(m_flow_new.log_likelihood()))
        nlp = np.append(nlp, np.array(m_flow_new.log_posterior_density()))
    
    pickle.dump(nll, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nll_"+target+".pkl", mode="wb"))
    pickle.dump(nlp, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nlp_"+target+".pkl", mode="wb"))

In [8]:
truedata_style='glm'
grw_setting='sep'
target='flow'
opt_percentage = 6
rep = 100
popu_dataset = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/"+truedata_style+"_mle_true_population.pkl", mode="rb"))
models_true = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/gp_"+truedata_style+"mle_mle_models.pkl", mode="rb"))
constrained_samples = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/constrained_samples_"+target+".pkl", mode="rb"))
nlp = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nlp_"+target+".pkl", mode="rb"))

In [10]:
IPM_pret_flow = Perted_IPM(popu_data=popu_dataset, GPmodel_true=models_true, 
                          grw_setting=grw_setting, target='m_flow_poi', truedata_style=truedata_style,
                          mcmc_para_sample=constrained_samples, summary='Full', nlog_post=nlp, opt_percentage=opt_percentage)

In [13]:
popu_opt_mode = 'OFF'

In [14]:

if popu_opt_mode == 'ON':
    print('\n\n\n' + 'Re-calculating summary_data for the opt MCMC samples' + '\n\n\n')
    summary_opt = IPM_pret_flow.simuVSsimu_singleModel_fun_parallel(rep=rep, random_seed=1,
                                        interested_models='Opt', evaluate_at_training=False)

    pickle.dump(summary_opt, 
                open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_opt_"+target+".pkl", mode="wb"))
    
    summary_around_opt = IPM_pret_flow.simuVSsimu_fun_parallel(rep=rep, random_seed=1, 
                                    opt_percentage=opt_percentage, interested_models='Opt', 
                                    MCMC_boolean_list='Opt', evaluate_at_training=False)
    pickle.dump(summary_around_opt, 
                open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_around_opt_"+target+".pkl", mode="wb"))
    
elif popu_opt_mode == 'OFF':
    print('\n\n\n' + 'Loading summary_data for the opt MCMC samples' + '\n\n\n')
    summary_opt = pickle.load(open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_opt_"+target+".pkl", mode="rb"))
    summary_around_opt = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_around_opt_"+target+".pkl", mode="rb"))




Re-calculating summary_data for the opt MCMC samples







 Parallel computing is starting 
 with repeat time 100 and random seed 1.






2024-01-11 14:38:21,795	INFO worker.py:1673 -- Started a local Ray instance.






 Parallel computing is starting 
 with repeat time 100 and random seed 1.






In [27]:
print('\n\n\n' + 'Re-calculating the top summary stats' + '\n\n\n')
num_mcmc = IPM_pret_flow.mcmc_para_sample[0].shape[0]

most_freq_summary_stats = pd.DataFrame(data=0.0, 
                                        index=range(np.sum(IPM_pret_flow.whether_around_opt_comp)), 
                                        columns=IPM_pret_flow.col_names)

for j in range(np.sum(IPM_pret_flow.whether_around_opt_comp)):
    d = summary_around_opt.loc[(0+j*rep):(rep-1+j*rep)].reset_index(drop=True).copy()
    auc0 = np.zeros(33)
    auc1 = np.zeros(33)
    for i in range(33):        
        fpr0, tpr0, _ = roc_curve(y_true=np.append(np.repeat(1, rep), np.repeat(0, rep)), 
                                y_score=np.append(summary_opt.iloc[:, i], d.iloc[:, i]), pos_label=0)
        auc0[i] = auc(fpr0, tpr0)
    
    most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.8]] = most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.8]]+ 1
    




Re-calculating the top summary stats





In [28]:
#0.75
print(np.sort(most_freq_summary_stats.sum())[-10:])
most_columns = most_freq_summary_stats.columns[np.argsort(most_freq_summary_stats.sum())[-10:]]
print(most_columns)

[ 0.  0.  0.  0.  1.  1. 29. 60. 85. 91.]
Index(['4b', '4a', '1g', '1f', 'raw_size', '1e', '7a', '7b', 'raw_flow',
       '16b'],
      dtype='object')
